# Video Saliency Prediction — end-to-end pipeline

This notebook trains a saliency model, runs it on the test videos, scores it with
**SIM / CC / NSS**, and renders an overlay GIF. It uses the `saliency` package in
this repository.

## Dataset

This notebook does **not** download anything. Unzip your local dataset so the
repo contains:

```
data/public_tests/
├── 01_test_file_input/{train,test}/<video>/source.mp4
└── 01_test_file_gt/{train,test}/<video>/{NNNN.png, fixations/NNNN.png}
```

Then set `DATA_ROOT` below to that folder (relative or absolute path). `NNNN.png`
is the ground-truth saliency for frame *N* (1-indexed); `fixations/NNNN.png` is
the binary fixation map used for NSS.

In [ ]:
import os
import torch

from saliency.data import SaliencyFrameDataset, InfiniteSampler
from saliency.models import SaliencyModel, ImprovedSaliencyModel
from saliency.losses import kld_loss, CombinedSaliencyLoss
from saliency.engine import train, EarlyStopping, WarmupCosineLR
from saliency.evaluate import SaliencyEvaluator
from saliency.metrics import calculate_metrics
from saliency.utils import set_seed, count_parameters

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Configuration

Choose the model and where the dataset lives. Increase `NUM_EPOCHS` / `NUM_STEPS` for real training (the defaults are small so the notebook runs quickly).

In [ ]:
DATA_ROOT = "data/public_tests"   # <-- point this at your unzipped dataset
MODEL = "improved"                # "baseline" or "improved"
PRETRAINED = True                 # ImageNet-pretrained backbone (needs internet on first run)

NUM_EPOCHS = 5
NUM_STEPS = 200
BATCH_SIZE = 8
LR = 3e-4
CHECKPOINT = "saliency.pth"

INPUT_ROOT = os.path.join(DATA_ROOT, "01_test_file_input")
GT_ROOT = os.path.join(DATA_ROOT, "01_test_file_gt")

## Datasets and loaders

Training uses an infinite sampler (a fixed number of random steps per epoch); validation iterates the test split once.

In [ ]:
train_ds = SaliencyFrameDataset(os.path.join(INPUT_ROOT, "train"),
                                os.path.join(GT_ROOT, "train"), augment=True)
val_ds = SaliencyFrameDataset(os.path.join(INPUT_ROOT, "test"),
                              os.path.join(GT_ROOT, "test"), augment=False)
print(f"Train frames: {len(train_ds)} | Val frames: {len(val_ds)}")

train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=BATCH_SIZE, sampler=InfiniteSampler(train_ds),
    num_workers=4, pin_memory=True, drop_last=True)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=2)

## Build the model

In [ ]:
if MODEL == "baseline":
    model = SaliencyModel(pretrained=PRETRAINED).to(device)
    criterion = kld_loss
else:
    model = ImprovedSaliencyModel(pretrained=PRETRAINED).to(device)
    criterion = CombinedSaliencyLoss().to(device)

print(f"Model: {MODEL} | trainable params: {count_parameters(model) / 1e6:.2f}M")

## Train

The best checkpoint (lowest validation loss) is written to `saliency.pth`.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = WarmupCosineLR(optimizer, base_lr=LR, warmup_epochs=2, total_epochs=NUM_EPOCHS)
early_stopping = EarlyStopping(patience=10, mode="min")

model, history = train(
    model, train_loader, val_loader, optimizer, criterion, device,
    num_epochs=NUM_EPOCHS, num_steps=NUM_STEPS, scheduler=scheduler,
    early_stopping=early_stopping, save_path=CHECKPOINT,
)

## Predict saliency for the test videos

Writes one PNG per frame to `outputs/<video>/NNNN.png`.

In [ ]:
evaluator = SaliencyEvaluator.from_checkpoint(model, CHECKPOINT, device)

test_input = os.path.join(INPUT_ROOT, "test")
for video in sorted(os.listdir(test_input)):
    evaluator.evaluate(os.path.join(test_input, video, "source.mp4"),
                       os.path.join("outputs", video))
print("Saved predictions to outputs/")

## Score: SIM / CC / NSS

In [ ]:
metrics = calculate_metrics("outputs", os.path.join(GT_ROOT, "test"))
for name, value in metrics.items():
    print(f"{name}: {value:.4f}")

## Qualitative overlay (optional)

Render a saliency-overlay GIF for one video and display it inline.

In [ ]:
import cv2
import imageio
import numpy as np
from saliency.utils import normalize_map, padding

video_id = sorted(os.listdir(test_input))[0]
cap = cv2.VideoCapture(os.path.join(test_input, video_id, "source.mp4"))
pred_dir = os.path.join("outputs", video_id)

with imageio.get_writer("out.gif", mode="I") as writer:
    for name in sorted(os.listdir(pred_dir)):
        if not name.endswith(".png"):
            continue
        ok, frame = cap.read()
        if not ok:
            break
        pred = cv2.imread(os.path.join(pred_dir, name), cv2.IMREAD_GRAYSCALE)
        heat = ((1 - padding(normalize_map(pred), frame.shape[0], frame.shape[1])) * 255).astype(np.uint8)
        heatmap = cv2.applyColorMap(heat, cv2.COLORMAP_JET)
        blended = np.clip(0.7 * frame + 0.3 * heatmap, 0, 255).astype(np.uint8)
        writer.append_data(cv2.cvtColor(blended, cv2.COLOR_BGR2RGB))
cap.release()

from IPython.display import Image
Image(open("out.gif", "rb").read())